# Environment Setup

This notebook creates the Unity Catalog infrastructure needed for the lab and generates the initial synthetic data.

In [0]:
%run ./variables

In [0]:
# 1. Create Catalog and Schemas
# Setting up the three-level namespace structure.


# Create catalog
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG_NAME}")

# Create schemas
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SCHEMA}")

# 2. Create Volumes
#  Volumes are needed for raw file storage and streaming checkpoints.
# Landing volume for raw files
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG_NAME}.{SCHEMA}.raw_data")

# System volume for checkpoints
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG_NAME}.{SCHEMA}.checkpoints")

print(f"✅ Unity Catalog structure created for {CATALOG_NAME}")






In [0]:
# Clean up existing tables from previous runs
for table in [TABLE_BRONZE_TRANSACTIONS, TABLE_SILVER_TRANSACTIONS, TABLE_SILVER_ALERTS,
              TABLE_SILVER_CUSTOMERS, TABLE_SILVER_MERCHANTS,
              TABLE_GOLD_DAILY_SUMMARY, TABLE_GOLD_SAR_PREFILL]:
    spark.sql(f"DROP TABLE IF EXISTS {table}")

# Clean up existing data in volumes if any
dbutils.fs.rm(LANDING_VOLUME_PATH, recurse=True)
dbutils.fs.rm(CHECKPOINT_PATH, recurse=True)


In [0]:
# Create landing subdirectories
dbutils.fs.mkdirs(TRANSACTIONS_RAW_PATH)
dbutils.fs.mkdirs(CUSTOMERS_RAW_PATH)
dbutils.fs.mkdirs(MERCHANTS_RAW_PATH)

In [0]:
# Run the data generator in bootstrap mode
dbutils.widgets.text("action", "bootstrap")

In [0]:
%run ./data_generator

In [0]:
# 4. Final Confirmation
# Verify that the infrastructure is ready.

# COMMAND ----------

# Quick check on landing files
try:
    files = dbutils.fs.ls(TRANSACTIONS_RAW_PATH)
    print(f"✅ Found {len(files)} transaction batch files in landing.")
except:
    print("❌ Transaction landing files not found!")

print("🎉 Environment setup complete!")